**PDF Webscraping**

This notebook is the first script of the project. It enables to download the pdf files from the initial website. It ensures to include new features (new pdf links, new pages)

In [2]:
# Imports
import os
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin, unquote
import re

# website
base_url = "https://goam.gov.ye"

# store pdf
output_dir = "../data/raw/pdf"
os.makedirs(output_dir, exist_ok=True)


# get all the pages with looted artefacts (current AND future)

# requests
url = "https://goam.gov.ye/Looted/"
response = requests.get(url)
response.raise_for_status() #error display

soup = BeautifulSoup(response.text, "html.parser")

# list and count pages (bottom buttons)
page_numbers = [1]
for elem in soup.find_all("a", href=True):
    href = elem["href"]
    match = re.search(r"Looted\?page=(\d+)", href)
    if match:
        page_numbers.append(int(match.group(1)))

max_page = max(page_numbers)

print(max_page)

#list of "looted pages"
pages = ["https://goam.gov.ye/Looted/"] + [f"https://goam.gov.ye/Looted?page={i}" for i in range(2, max_page + 1)]

# list pdf files (through links)
pdf_links = []

for page in pages:
    r = requests.get(page)
    r.raise_for_status()

    soup = BeautifulSoup(r.text, "html.parser")

    links = [
        a["href"]
        for a in soup.find_all("a", href=True)
        if "/Home/DownloadLostFile/" in a["href"]
    ]

    pdf_links.extend(links)

# print(f"{len(pdf_links)} PDF links found")

# Delete duplicates (more than expected)
pdf_links = list(dict.fromkeys(pdf_links))
print(f"{len(pdf_links)} pdf files")

# Download
for link in pdf_links:
    pdf_url = urljoin(base_url, link)
    pdf_id = link.split("/")[-1]
    response = requests.get(pdf_url)
    response.raise_for_status()

    # the file name does not appear in the link 
    cd = response.headers.get("Content-Disposition")
    if cd and "filename=" in cd:
        filename = cd.split("filename=")[-1].strip(' "')
    else:
        filename = f"file_{pdf_id}.pdf"

    # File name cleaning
    filename = unquote(filename)
    filename = re.sub(r'[<>:"/\\|?*]', "_", filename)
    filename = filename.strip()

    def sanitize_filename(name: str) -> str:
    # Keep only the filename, without directories
    name = Path(name).name.strip()

    # If the name comes from a header-like string, keep the UTF-8 filename part
    marker = "filename_=UTF-8''"
    if marker in name:
        name = name.split(marker, 1)[1]
    elif ";" in name:
        # Otherwise keep only the part before a possible ';'
        name = name.split(";", 1)[0].strip()

    # Decode URL-encoded characters
    name = unquote(name)

    # Replace filesystem-invalid characters
    name = re.sub(r'[<>:"/\\|?*]', "_", name)

    return name.strip()

    file_path = os.path.join(output_dir, filename)
    with open(file_path, "wb") as f:
        f.write(response.content)
    print(f"{filename} downloaded!")

2
31 pdf files
31-______ _________31.pdf_; filename_=UTF-8''31-آثارنا المنهوبة_31.pdf downloaded!
0-______ ________ 30.pdf_; filename_=UTF-8''0-آثارنا المنهوبة 30.pdf downloaded!
29-______ _________29.pdf_; filename_=UTF-8''29-آثارنا المنهوبة_29.pdf downloaded!
0-______ _________28.pdf_; filename_=UTF-8''0-آثارنا المنهوبة_28.pdf downloaded!
0-______ _________27.pdf_; filename_=UTF-8''0-آثارنا المنهوبة_27.pdf downloaded!
0-______ _________26.pdf_; filename_=UTF-8''0-آثارنا المنهوبة_26.pdf downloaded!
0-______ _________25.pdf_; filename_=UTF-8''0-آثارنا المنهوبة_25.pdf downloaded!
0-______ _________24.pdf_; filename_=UTF-8''0-آثارنا المنهوبة_24.pdf downloaded!
0-______ _________23.pdf_; filename_=UTF-8''0-آثارنا المنهوبة_23.pdf downloaded!
0-______ _________22.pdf_; filename_=UTF-8''0-آثارنا المنهوبة_22.pdf downloaded!
0-______ _________21.pdf_; filename_=UTF-8''0-آثارنا المنهوبة_21.pdf downloaded!
0-______ _________20.pdf_; filename_=UTF-8''0-آثارنا المنهوبة_20.pdf downloaded!
0-______ 